# Algorithmic Fairness: Impact of Model Selection Metrics

🇺🇸 **[EN]** This repository contains the experimental pipeline for the paper:
**"The Impact of Model Selection Metrics during Hyperparameter Tuning on Algorithmic Fairness: An Empirical Study"**

🇧🇷 **[PT-BR]** Este repositório contém o pipeline experimental do artigo:
**"The Impact of Model Selection Metrics during Hyperparameter Tuning on Algorithmic Fairness: An Empirical Study"** (*O impacto das métricas de seleção de modelos durante o ajuste de hiperparâmetros na equidade algorítmica: Um estudo empírico*)

---

### Abstract / Resumo

🇺🇸 **[EN]** This study investigates how the choice of optimization metric (e.g., Accuracy, Recall, PR-AUC) during hyperparameter tuning shapes error distribution and algorithmic fairness. The pipeline evaluates five high-stakes datasets and compares two tree-based algorithms (Random Forest and Gradient Boosting). To ensure methodological rigor and prevent data leakage, all experiments are conducted using a strictly controlled Nested Stratified Cross-Validation setup.

🇧🇷 **[PT-BR]** Este estudo investiga como a escolha da métrica de otimização (ex: Accuracy, Recall, PR-AUC) durante o ajuste de hiperparâmetros molda a distribuição de erros e a equidade algorítmica (*fairness*). O pipeline avalia cinco bases de dados de domínios críticos e compara dois algoritmos de aprendizado (Random Forest e Gradient Boosting). Para garantir rigor metodológico e evitar vazamento de dados, todos os experimentos são conduzidos utilizando uma arquitetura controlada de Validação Cruzada Aninhada e Estratificada (*Nested Stratified Cross-Validation*).

---

### Environment Setup / Configuração do Ambiente

🇺🇸 **[EN]** To ensure exact reproducibility and avoid dependency conflicts, we strongly recommend running this pipeline in an isolated virtual environment (e.g., `venv` or `conda`). All necessary Python libraries, including data processing and statistical testing packages, are listed in the `requirements.txt` file. 

🇧🇷 **[PT-BR]** Para garantir a reprodutibilidade exata e evitar conflitos de dependências, recomendamos fortemente a execução deste pipeline em um ambiente virtual isolado (ex: `venv` ou `conda`). Todas as bibliotecas Python necessárias, incluindo pacotes de processamento de dados e testes estatísticos, estão listadas no arquivo `requirements.txt`.

**Installation / Instalação:**

```bash
git clone https://github.com/mlab-inf-ufrgs/semish2026_OptimizationMetrics-Bias.git

cd semish2026_OptimizationMetrics-Bias

pip install -r requirements.txt
```

---

## 1. Library Imports and Global Settings / Importação de Bibliotecas e Configurações Globais

🇺🇸 **[EN]** In this initial step, we import all necessary modules for data manipulation, statistical analysis, and machine learning modeling. We also establish the global parameters strictly necessary for exact reproducibility, such as fixing the `RANDOM_STATE` seed, isolating output directories, and defining the official custom color palette used to generate the paper's visualizations.

🇧🇷 **[PT-BR]** Nesta etapa inicial, importamos todos os módulos necessários para manipulação de dados, análise estatística e modelagem computacional. Também estabelecemos os parâmetros globais estritamente necessários para a reprodutibilidade exata, como a fixação da semente `RANDOM_STATE` e o isolamento dos diretórios de saída.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import random
from scipy.stats import entropy, ks_2samp, friedmanchisquare
import scikit_posthocs as sp

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    matthews_corrcoef, make_scorer, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
DATASETS_DIR = 'datasets'
OUTPUT_DIR = 'results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Dataset Metadata and Fairness Configurations / Metadados e Configurações de Equidade dos Datasets

🇺🇸 **[EN]** In this section, we define the metadata for the five benchmark datasets evaluated in the study: COMPAS, Heart, Adult, Dropout, and Intersectional Bias. To ensure accurate and reproducible fairness assessments, we statically map the target variable, the sensitive attribute, the protected group value, and the domain-specific "favorable class" (e.g., predicting "no recidivism" or "no disease" as the positive outcome). We also specify the removal of one-hot encoded redundant columns to isolate the binary protected attribute and prevent data leakage during modeling.

🇧🇷 **[PT-BR]** Nesta seção, definimos os metadados para as cinco bases de dados de *benchmark* avaliadas no estudo: COMPAS, Heart, Adult, Dropout e Intersectional Bias. Para garantir avaliações de justiça (*fairness*) precisas e reprodutíveis, mapeamos estaticamente a variável alvo, o atributo sensível, o valor do grupo protegido e a "classe favorável" específica do domínio (ex: prever "não reincidência" ou "sem doença" como o resultado positivo). Também especificamos a remoção de colunas redundantes (geradas por *one-hot encoding*) para isolar o atributo protegido binário e evitar vazamento de dados durante a modelagem.

In [2]:
DATASET_CONFIGS = {
    'compas-scores-raw_converted.csv': {
        'target': 'is_recid',
        'sensitive': 'race_African-American',
        'protected_value': True,
        'favorable_class': 0,
        'drop': ['race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Native American', 'race_Other']
    },
    'heart_converted.csv': {
        'target': 'target',
        'sensitive': 'sex',
        'protected_value': 0,
        'favorable_class': 0 
    },
    'adult_converted.csv': {
        'target': 'income',
        'sensitive': 'race_White',
        'protected_value': False,
        'favorable_class': 1,
        'drop': ['race_Amer-Indian-Eskimo', 'race_Asian-Pac-Islander', 'race_Black', 'race_Other']
    },
    'dropout_converted.csv': {
        'target': 'Target',
        'sensitive': 'Gender',
        'protected_value': 0,
        'favorable_class': 0 
    },
    'intersectional-bias_converted.csv': {
        'target': 'Diagnosis',
        'sensitive': 'Sex',
        'protected_value': 0, 
        'favorable_class': 0, 
        'drop': ['Race_Asian', 'Race_Black', 'Race_Hispanic', 'Race_White']
    }
}

## 3. Fairness Metrics Formulation / Formulação das Métricas de Justiça

🇺🇸 **[EN]** In this section, we implement the core mathematical formulations for assessing algorithmic fairness, explicitly aligned with the methodology described in the paper. We calculate the pre-training bias (baseline Disparate Impact) to establish the structural inequities present in the raw data. Post-training disparities are quantified using three complementary metrics:
* **Disparate Impact (DI):** Measures the ratio of favorable predicted outcomes between unprivileged and privileged groups.
* **Sensitivity Gap:** Computes the difference in true positive rates (TPR) between groups. Negative values indicate the model is more capable of detecting positives for the privileged group.
* **Average Absolute Odds Difference (AAOD):** Provides a comprehensive view of error asymmetry by averaging the absolute differences in both False Positive Rates (FPR) and True Positive Rates (TPR) across groups.

*To ensure mathematical integrity, these calculations do not dynamically infer positive labels; they strictly use the domain-specific `favorable_class` established in the dataset configuration.*

🇧🇷 **[PT-BR]** Nesta seção, implementamos as formulações matemáticas centrais para avaliar a justiça algorítmica, alinhadas explicitamente com a metodologia descrita no artigo. Calculamos o viés pré-treinamento (*baseline Disparate Impact*) para estabelecer as desigualdades estruturais presentes nos dados brutos. As disparidades pós-treinamento são quantificadas usando três métricas complementares:
* **Disparate Impact (DI):** Mede a razão de previsões favoráveis entre grupos não privilegiados e privilegiados.
* **Sensitivity Gap:** Calcula a diferença nas taxas de verdadeiros positivos (TPR) entre os grupos. Valores negativos indicam que o modelo é mais capaz de detectar positivos para o grupo privilegiado.
* **Average Absolute Odds Difference (AAOD):** Fornece uma visão abrangente da assimetria de erros, calculando a média das diferenças absolutas tanto nas Taxas de Falsos Positivos (FPR) quanto nas Taxas de Verdadeiros Positivos (TPR) entre os grupos.

*Para garantir a integridade matemática, esses cálculos não inferem os rótulos positivos dinamicamente; eles utilizam estritamente a `favorable_class` de domínio específico estabelecida na configuração da base de dados.*

In [3]:
def calculate_specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0

def calculate_specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0

def calculate_fairness_metrics(y_true, y_pred, sensitive_attr, favorable_class, pos_label_for_recall=1):
    df = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred, 'sensitive': sensitive_attr})
    prot, priv = 0, 1

    df_prot = df[df['sensitive'] == prot]
    df_priv = df[df['sensitive'] == priv]
    
    sr_prot = df_prot['y_pred'].mean() if len(df_prot) > 0 else 0
    sr_priv = df_priv['y_pred'].mean() if len(df_priv) > 0 else 0
    
    if sr_priv == 0 and sr_prot == 0:
        di = 1.0
    else:
        raw_di = sr_prot / sr_priv if sr_priv > 0 else float('inf')
        di = min(raw_di, 1 / raw_di) if raw_di > 0 else 0.0

    def get_rates(y_t, y_p):
        if len(y_t) == 0: return 0, 0
        cm = confusion_matrix(y_t, y_p, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        return fpr, tpr

    fpr_prot, tpr_prot = get_rates(df_prot['y_true'], df_prot['y_pred'])
    fpr_priv, tpr_priv = get_rates(df_priv['y_true'], df_priv['y_pred'])

    aaod = 0.5 * (abs(fpr_prot - fpr_priv) + abs(tpr_prot - tpr_priv))

    recall_prot = recall_score(df_prot['y_true'], df_prot['y_pred'], pos_label=pos_label_for_recall, zero_division=0)
    recall_priv = recall_score(df_priv['y_true'], df_priv['y_pred'], pos_label=pos_label_for_recall, zero_division=0)
    sensitivity_gap = recall_prot - recall_priv

    return {
        "Sensitivity Gap": sensitivity_gap,
        "Disparate Impact": di,
        "AAOD": aaod,
        "Recall Unprivileged": recall_prot,
        "Recall Privileged": recall_priv
    }

In [4]:
def calculate_pretraining_bias(df, config, name):
    target, sensitive, prot_val = config['target'], config['sensitive'], config['protected_value']
    favorable_class = config['favorable_class']

    s = (df[sensitive] != prot_val).astype(int) 
    y = df[target].astype(int).values

    counts = pd.Series(y).value_counts(normalize=True) * 100
    class_dist = f"{counts.get(1, 0):.1f} / {counts.get(0, 0):.1f}"
    ci_ratio = max(counts) / min(counts) if min(counts) > 0 else 1.0

    # Disparate Impact (DI): P(Y=favorable|Unprivileged) / P(Y=favorable|Privileged)
    prob_prot = np.mean(y[s == 0] == favorable_class)
    prob_priv = np.mean(y[s == 1] == favorable_class)
    di = prob_prot / prob_priv if prob_priv > 0 else 1.0

    # Kullback-Leibler (KL) Divergence
    kl = entropy([1 - prob_prot, prob_prot], [1 - prob_priv, prob_priv])

    # Kolmogorov-Smirnov (KS) Average
    num_features = df.select_dtypes(include=[np.number]).columns.drop([target], errors='ignore')
    ks_values = [ks_2samp(df[s == 0][col].dropna(), df[s == 1][col].dropna())[0] for col in num_features]
    avg_ks = np.mean(ks_values) if ks_values else 0.0

    return {
        "Dataset": name.replace('_converted.csv', ''),
        "N": len(df),
        "Class Dist (%)": class_dist,
        "CI Ratio": round(ci_ratio, 2),
        "DI (Pre-train)": round(di, 2),
        "KL": round(kl, 3),
        "KS (Avg)": round(avg_ks, 3)
    }


## 4. Experimental Pipeline (Nested Cross-Validation) / Pipeline Experimental (Validação Cruzada Aninhada)

🇺🇸 **[EN]** This section defines the core experimental pipeline. To ensure rigorous evaluation and completely prevent data leakage, the data preprocessing step (imputation and standardization) is strictly isolated within the cross-validation loop using a Scikit-Learn `Pipeline`. We employ a Stratified Nested Cross-Validation scheme. The outer loop evaluates the models' generalization capability, while the inner loop is dedicated to hyperparameter tuning using a `RandomizedSearchCV` focused on maximizing specific performance metrics (e.g., Accuracy, PR-AUC, Sensitivity). 

🇧🇷 **[PT-BR]** Esta seção define o pipeline experimental central. Para garantir uma avaliação rigorosa e evitar completamente o vazamento de dados (*data leakage*), a etapa de pré-processamento (imputação e padronização) é estritamente isolada dentro do loop de validação cruzada utilizando um `Pipeline` do Scikit-Learn. Empregamos um esquema de Validação Cruzada Aninhada e Estratificada (*Nested Stratified Cross-Validation*). O loop externo avalia a capacidade de generalização dos modelos, enquanto o loop interno é dedicado ao ajuste de hiperparâmetros usando um `RandomizedSearchCV` focado em maximizar métricas de desempenho específicas (ex: Acurácia, PR-AUC, Sensibilidade).

In [5]:
def run_rigorous_pipeline():
    pre_bias_results = []
    exp_results = []
    raw_predictions = []

    if not os.path.exists(DATASETS_DIR):
        print(f"Erro: Pasta '{DATASETS_DIR}' não encontrada.")
        return None, None, None

    files = sorted([f for f in os.listdir(DATASETS_DIR) if f.endswith('.csv')])

    models = {
        'Random Forest': (RandomForestClassifier(random_state=RANDOM_STATE), {
            'classifier__n_estimators': [50, 100, 250], 'classifier__max_depth': [10, 20, None]
        }),
        'Gradient Boosting': (GradientBoostingClassifier(random_state=RANDOM_STATE), {
            'classifier__n_estimators': [100, 200], 'classifier__learning_rate': [0.1, 0.2], 'classifier__max_depth': [3, 5]
        })
    }

    scoring_metrics = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall', 
        'specificity': make_scorer(calculate_specificity),
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision',
        'mcc': make_scorer(matthews_corrcoef)
    }

    for filename in files:
        if filename not in DATASET_CONFIGS: continue
        print(f"\n>>> Processing: {filename}")
        df = pd.read_csv(os.path.join(DATASETS_DIR, filename))

        config = DATASET_CONFIGS[filename]
        
        favorable_class = config['favorable_class']
        
        pre_bias_results.append(calculate_pretraining_bias(df, config, filename))

        target_col = config['target']
        sensitive_col = config['sensitive']
        prot_val = config['protected_value']
        favorable_class = config['favorable_class']
        if 'drop' in config: df = df.drop(columns=config['drop'])

        y = LabelEncoder().fit_transform(df[target_col].astype(str))
        s = (df[sensitive_col] != prot_val).astype(int)
        X = df.drop(columns=[target_col])
        y_s = y.astype(str) + "_" + s.astype(str)
        
        outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

        for model_name, (model_obj, param_grid) in models.items():
            for metric_name, scorer in scoring_metrics.items():
                print(f"  {model_name} | Optimized metric: {metric_name}...")

                for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y_s)):
                    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                    y_train, y_test = y[train_idx], y[test_idx]
                    s_test = s[test_idx]
                    y_s_train = y_s[train_idx]

                    categorical_cols = X.select_dtypes(include=['object', 'bool']).columns
                    numerical_cols = X.select_dtypes(include=[np.number]).columns

                    preprocessor = ColumnTransformer(
                        transformers=[
                            ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numerical_cols),
                            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
                        ]
                    )

                    pipeline = Pipeline([
                        ('preprocessor', preprocessor),
                        ('classifier', model_obj)
                    ])

                    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
                    search = RandomizedSearchCV(pipeline, param_grid, n_iter=30, scoring=scorer,
                                                cv=list(inner_cv.split(X_train, y_s_train)),
                                                random_state=RANDOM_STATE, n_jobs=-1)
                    search.fit(X_train, y_train)

                    y_pred = search.best_estimator_.predict(X_test)

                    y_test_vals = y_test.values if hasattr(y_test, 'values') else y_test
                    s_test_vals = s_test.values if hasattr(s_test, 'values') else s_test

                    for i in range(len(y_pred)):
                        raw_predictions.append({
                            'Dataset': filename,
                            'Model': model_name,
                            'Optimized_Metric': metric_name,
                            'Fold': fold_idx,
                            'True_Label': y_test_vals[i],
                            'Predicted_Label': y_pred[i],
                            'Sensitive_Attribute': s_test_vals[i]
                        })

                    unfavorable_class = 1 if favorable_class == 0 else 0
                    perf = {
                        'Accuracy': accuracy_score(y_test, y_pred), 
                        'Recall': recall_score(y_test, y_pred, pos_label=favorable_class, zero_division=0),
                        'Precision': precision_score(y_test, y_pred, pos_label=favorable_class, zero_division=0), 
                        'Specificity': recall_score(y_test, y_pred, pos_label=unfavorable_class, zero_division=0),
                        'MCC': matthews_corrcoef(y_test, y_pred)
                    }
                    
                    fair = calculate_fairness_metrics(
                        y_true=y_test, 
                        y_pred=y_pred, 
                        sensitive_attr=s_test, 
                        favorable_class=favorable_class
                    )

                    res = {'Dataset': filename, 'Model': model_name, 'Optimized_Metric': metric_name, 'Fold': fold_idx}
                    res.update(perf); res.update(fair); exp_results.append(res)
 
    df_pre_bias = pd.DataFrame(pre_bias_results)
    df_exp = pd.DataFrame(exp_results)
    df_raw = pd.DataFrame(raw_predictions)

    df_pre_bias.to_csv(os.path.join(OUTPUT_DIR, 'table_1_pre_bias.csv'), index=False)
    df_exp.to_csv(os.path.join(OUTPUT_DIR, 'final_results.csv'), index=False)
    df_raw.to_csv(os.path.join(OUTPUT_DIR, 'raw_results.csv'), index=False)
    
    return df_pre_bias, df_exp, df_raw

## 5. Statistical Analysis / Análise Estatística

🇺🇸 **[EN]** In this phase, we conduct rigorous statistical testing to determine if the observed differences in performance and fairness metrics across various optimization objectives are statistically significant. We employ the non-parametric Friedman test to compare the average ranks of the optimization metrics across the datasets. If the null hypothesis is rejected (p < 0.05), we proceed with the Nemenyi post-hoc test for pairwise comparisons. The results of these statistical tests are exported to corroborate the empirical claims made in the paper.

🇧🇷 **[PT-BR]** Nesta fase, conduzimos testes estatísticos rigorosos para determinar se as diferenças observadas nas métricas de desempenho e justiça (*fairness*) entre os diferentes objetivos de otimização são estatisticamente significativas. Empregamos o teste não paramétrico de Friedman para comparar os rankings médios das métricas de otimização ao longo das bases de dados. Se a hipótese nula for rejeitada (p < 0.05), prosseguimos com o teste *post-hoc* de Nemenyi para comparações pareadas. Os resultados desses testes estatísticos são exportados para corroborar as afirmações empíricas feitas no artigo.

In [6]:
def run_statistical_analysis(df_exp):
    summary = df_exp.groupby(['Dataset', 'Model', 'Optimized_Metric']).mean(numeric_only=True).reset_index()

    summary['|Sensitivity Gap|'] = summary['Sensitivity Gap'].abs()
    summary['|DI - 1|'] = (summary['Disparate Impact'] - 1).abs()

    metrics_to_test = [
        'AAOD', '|DI - 1|', '|Sensitivity Gap|',
        'Accuracy', 'Precision', 'Recall', 'Specificity', 'MCC'
    ]

    all_friedman_results = []
    all_nemenyi_matrices = []

    for metric in metrics_to_test:
        pivot_df = summary.pivot(index=['Dataset', 'Model'], columns='Optimized_Metric', values=metric).dropna()
        
        pivot_df.index = [f"{d}_{m}" for d, m in pivot_df.index]
        
        if pivot_df.shape[1] > 1:
            stat, p = friedmanchisquare(*[pivot_df[col] for col in pivot_df.columns])
            print(f"Teste Global (N={len(pivot_df)}) -> Statistic = {stat:.4f}, p-value = {p:.4e}")
            
            all_friedman_results.append({
                'Evaluated_Metric': metric,
                'Statistic': stat,
                'p-value': p
            })
            
            if p < 0.05:
                print("  -> Significant difference detected. Adding to the Nemenyi matrix...")
                nemenyi_df = sp.posthoc_nemenyi_friedman(pivot_df)
                nemenyi_df.columns, nemenyi_df.index = pivot_df.columns, pivot_df.columns
                
                nemenyi_df = nemenyi_df.reset_index().rename(columns={'index': 'Versus_Metric'})
                nemenyi_df.insert(0, 'Evaluated_Metric', metric)
                
                all_nemenyi_matrices.append(nemenyi_df)
            else:
                print("  -> No significant overall differences (p >= 0.05).")

    if all_friedman_results:
        df_friedman_final = pd.DataFrame(all_friedman_results)
        path_friedman = os.path.join(OUTPUT_DIR, 'table_friedman_consolidated.csv')
        df_friedman_final.to_csv(path_friedman, index=False)
        print(f"\n>>> Success! Friedman consolidated file saved in: {path_friedman}")

    if all_nemenyi_matrices:
        df_nemenyi_final = pd.concat(all_nemenyi_matrices, ignore_index=True)
        path_nemenyi = os.path.join(OUTPUT_DIR, 'table_nemenyi_consolidated.csv')
        df_nemenyi_final.to_csv(path_nemenyi, index=False)
        print(f">>> Success! Consolidated Nemenyi file saved to:{path_nemenyi}")

## 6. Pipeline Execution / Execução do Pipeline

🇺🇸 **[EN]** In this final section, we define the main execution block that orchestrates the entire experimental pipeline. It sequentially runs the nested cross-validation and conducts the non-parametric statistical significance tests.

🇧🇷 **[PT-BR]** Nesta seção final, definimos o bloco principal de execução que orquestra todo o pipeline experimental. Ele executa sequencialmente a validação cruzada aninhada e conduz os testes não paramétricos de significância estatística.

In [8]:
if __name__ == "__main__":
    df_pre_bias, df_exp, df_raw = run_rigorous_pipeline()

    if df_exp is not None:

        run_statistical_analysis(df_exp)

        print("\n>>> Pipeline completed.")


>>> Processing: adult_converted.csv
  Random Forest | Optimized metric: accuracy...
  Random Forest | Optimized metric: precision...
  Random Forest | Optimized metric: recall...
  Random Forest | Optimized metric: specificity...
  Random Forest | Optimized metric: roc_auc...
  Random Forest | Optimized metric: pr_auc...
  Random Forest | Optimized metric: mcc...
  Gradient Boosting | Optimized metric: accuracy...
  Gradient Boosting | Optimized metric: precision...
  Gradient Boosting | Optimized metric: recall...
  Gradient Boosting | Optimized metric: specificity...
  Gradient Boosting | Optimized metric: roc_auc...
  Gradient Boosting | Optimized metric: pr_auc...
  Gradient Boosting | Optimized metric: mcc...

>>> Processing: compas-scores-raw_converted.csv
  Random Forest | Optimized metric: accuracy...
  Random Forest | Optimized metric: precision...
  Random Forest | Optimized metric: recall...
  Random Forest | Optimized metric: specificity...
  Random Forest | Optimized metr